# Notebook 8 – Greenhouse Gas Emissions Data Processing

## Purpose

This notebook establishes a reproducible processing workflow for greenhouse gas emissions data used by the CANOE geospatial preprocessing pipeline.

Rather than relying on legacy Excel input files with undocumented units or unknown provenance, this notebook reconstructs the emissions input tables directly from official Government of Canada datasets. The objective is to ensure that all emissions data entering the model are fully traceable, unit-consistent, and reproducible.

## Objectives

- Load the official 2024 Greenhouse Gas Emissions from Large Facilities dataset.
- Compare the CSV and GeoJSON versions of the dataset.
- Identify the emissions fields, reporting units, and metadata.
- Validate facility counts, coordinate systems, and geographic coverage.
- Document all unit conversions and processing assumptions.
- Prepare a standardized emissions dataset for subsequent spatial aggregation to CANOE regions.
- Export cleaned emissions tables for use in the SQL encoding workflow.

## Outputs

The outputs generated by this notebook are intended to serve as standardized intermediate datasets for downstream preprocessing modules and should not require manual editing.

In [1]:
# =============================================================================
# Imports and path configuration
# =============================================================================

from pathlib import Path
import json

import pandas as pd
import geopandas as gpd


# -----------------------------------------------------------------------------
# Dataset configuration
# -----------------------------------------------------------------------------

DATASET_YEAR = 2024
DATASET_NAME = "co2_large_facilities"


# -----------------------------------------------------------------------------
# Project paths
# -----------------------------------------------------------------------------

PROJECT_ROOT = Path(r"C:\Users\aviga\Research\repos\temoa_geospace")

RAW_DIR = PROJECT_ROOT / "data_files" / "raw" / "emissions"
PROCESSED_DIR = PROJECT_ROOT / "data_files" / "processed" / "emissions"

CO2_OUTPUT_DIR = PROCESSED_DIR / f"{DATASET_NAME}_{DATASET_YEAR}"


# -----------------------------------------------------------------------------
# Source files
# -----------------------------------------------------------------------------

CO2_CSV_PATH = RAW_DIR / "Greenhouse gas emissions from large facilities - 2024.csv"
CO2_JSON_PATH = RAW_DIR / "AirEmissions_GHG_2024.json"


# -----------------------------------------------------------------------------
# Output files
# -----------------------------------------------------------------------------

CO2_CLEAN_CSV = CO2_OUTPUT_DIR / f"{DATASET_NAME}_{DATASET_YEAR}_clean.csv"
CO2_CLEAN_GPKG = CO2_OUTPUT_DIR / f"{DATASET_NAME}_{DATASET_YEAR}_clean.gpkg"
CO2_METADATA_CSV = CO2_OUTPUT_DIR / f"{DATASET_NAME}_{DATASET_YEAR}_metadata.csv"
CO2_COLUMN_AUDIT_CSV = CO2_OUTPUT_DIR / f"{DATASET_NAME}_{DATASET_YEAR}_column_audit.csv"


# -----------------------------------------------------------------------------
# Create required directories
# -----------------------------------------------------------------------------

for directory in [
    RAW_DIR,
    PROCESSED_DIR,
    CO2_OUTPUT_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


# -----------------------------------------------------------------------------
# Validate source files
# -----------------------------------------------------------------------------

required_files = [
    CO2_CSV_PATH,
    CO2_JSON_PATH,
]

missing_files = [path for path in required_files if not path.exists()]

if missing_files:
    for path in missing_files:
        print(f"Missing required input file: {path}")
    raise FileNotFoundError("One or more required CO2 input files are missing.")

print("Input files found:")
for path in required_files:
    print(f"  ✓ {path.name}")

print("\nOutput directory:")
print(f"  ✓ {CO2_OUTPUT_DIR}")

Input files found:
  ✓ Greenhouse gas emissions from large facilities - 2024.csv
  ✓ AirEmissions_GHG_2024.json

Output directory:
  ✓ C:\Users\aviga\Research\repos\temoa_geospace\data_files\processed\emissions\co2_large_facilities_2024


In [2]:
# =============================================================================
# Load and inspect source datasets
# =============================================================================

# -----------------------------------------------------------------------------
# Load CSV source
# -----------------------------------------------------------------------------

co2_csv = pd.read_csv(CO2_CSV_PATH)

print("CSV loaded successfully")
print(f"  Rows:    {co2_csv.shape[0]:,}")
print(f"  Columns: {co2_csv.shape[1]:,}")


# -----------------------------------------------------------------------------
# Load JSON / GeoJSON source
# -----------------------------------------------------------------------------

co2_geo = gpd.read_file(CO2_JSON_PATH)

print("\nJSON / GeoJSON loaded successfully")
print(f"  Rows:    {co2_geo.shape[0]:,}")
print(f"  Columns: {co2_geo.shape[1]:,}")
print(f"  CRS:     {co2_geo.crs}")


# -----------------------------------------------------------------------------
# Compare source dimensions
# -----------------------------------------------------------------------------

print("\nSource comparison")
print(f"  CSV rows:     {len(co2_csv):,}")
print(f"  GeoJSON rows: {len(co2_geo):,}")

if len(co2_csv) == len(co2_geo):
    print("  ✓ CSV and GeoJSON have the same number of records.")
else:
    print("  ⚠ CSV and GeoJSON have different record counts.")


# -----------------------------------------------------------------------------
# Preview both sources
# -----------------------------------------------------------------------------

display(co2_csv.head())
display(co2_geo.head())

CSV loaded successfully
  Rows:    1,879
  Columns: 18

JSON / GeoJSON loaded successfully
  Rows:    1,879
  Columns: 13
  CRS:     EPSG:4326

Source comparison
  CSV rows:     1,879
  GeoJSON rows: 1,879
  ✓ CSV and GeoJSON have the same number of records.


,Facility ID,Facility name,Company name,City,Address,Postal code,Province,Latitude,Longitude,Total emissions,Unit,Year,Report year,Industry classification,Industry classification link,Facility information,Facility details,More information
0,10001,Division Alma,Produits forestiers Résolu,Alma,1100 Melançon Street,G8B 5W2,Quebec,48.56500,-71.65556,53.34,kilotonnes of carbon dioxide equivalents (kt C...,2024,2026,Mechanical Pulp Mills,https://www23.statcan.gc.ca/imdb/p3VD.pl?Funct...,https://climate-change.canada.ca/facility-emis...,NaN,https://indicators-map.dev.ec.gc.ca/App/Detail...
1,10003,"Foothills Pipeline, Alberta",Foothills Pipe Lines Ltd.,Airdrie,NaN,T4A 2G7,Alberta,0.00000,0.00000,254.60,kilotonnes of carbon dioxide equivalents (kt C...,2024,2026,Pipeline Transportation of Natural Gas,https://www23.statcan.gc.ca/imdb/p3VD.pl?Funct...,https://climate-change.canada.ca/facility-emis...,NaN,https://indicators-map.dev.ec.gc.ca/App/Detail...
2,10004,Kingston CoGen,Kingston CoGen Limited Partnership,Bath,5146 Taylor-Kidd Boulevard,K0H 1G0,Ontario,44.20950,-76.72460,1.68,kilotonnes of carbon dioxide equivalents (kt C...,2024,2026,Fossil-Fuel Electric Power Generation,https://www23.statcan.gc.ca/imdb/p3VD.pl?Funct...,https://climate-change.canada.ca/facility-emis...,NaN,https://indicators-map.dev.ec.gc.ca/App/Detail...
3,10006,Redwater Fertilizer Operations,Nutrien (Canada) Holdings ULC,Sturgeon County,56225 SH643,T0A 2W0,Alberta,53.84200,-113.09300,1095.80,kilotonnes of carbon dioxide equivalents (kt C...,2024,2026,Chemical Fertilizer (except Potash) Manufacturing,https://www23.statcan.gc.ca/imdb/p3VD.pl?Funct...,https://climate-change.canada.ca/facility-emis...,NaN,https://indicators-map.dev.ec.gc.ca/App/Detail...
4,10007,Alberta Envirofuels,Keyera Corp,Edmonton,9511 17 Street Northwest,T6P 1Y3,Alberta,53.53199,-113.36492,315.20,kilotonnes of carbon dioxide equivalents (kt C...,2024,2026,Petrochemical Manufacturing,https://www23.statcan.gc.ca/imdb/p3VD.pl?Funct...,https://climate-change.canada.ca/facility-emis...,NaN,https://indicators-map.dev.ec.gc.ca/App/Detail...


,Name,Company,Facility type,City,Province,Latitude,Longitude,Total GHG emissions,Units,Year,Report year,Symbol,geometry
0,IMT Integrated Facility,IMT Standen's GP Inc.,Forging,Kitchener,Ontario,43.40108,-80.44677,9.5,kilotonnes of carbon dioxide equivalents (kt ...,2024,2026,1,POINT (-80.44677 43.40108)
1,Spec Furniture,Spec Furniture Inc.,Institutional Furniture Manufacturing,Toronto,Ontario,43.70430,-79.58770,0.6,kilotonnes of carbon dioxide equivalents (kt ...,2024,2026,1,POINT (-79.5877 43.7043)
2,LA BRASSERIE LABATT,Labatt Brewing Company Ltd,Breweries,Lasalle,Quebec,45.42550,-73.64470,6.1,kilotonnes of carbon dioxide equivalents (kt ...,2024,2026,1,POINT (-73.6447 45.4255)
3,BITUMAR,Bitumar Inc.,All Other Miscellaneous Manufacturing,Montréal,Quebec,45.63330,-73.51680,9.1,kilotonnes of carbon dioxide equivalents (kt ...,2024,2026,1,POINT (-73.5168 45.6333)
4,Golden Acre Farms Inc.,GOLDEN ACRE FARMS INC.,Other Food Crops Grown Under Cover,Kingsville,Ontario,42.05787,-82.69498,9.8,kilotonnes of carbon dioxide equivalents (kt ...,2024,2026,1,POINT (-82.69498 42.05787)


In [3]:
# =============================================================================
# Dataset metadata
# =============================================================================

DATASET_PROVIDER = "Environment and Climate Change Canada (ECCC)"
DATASET_TITLE = "Greenhouse Gas Emissions from Large Facilities"
DATASET_YEAR = 2024

EMISSIONS_UNIT = "kt CO2e/year"
GEOMETRY_TYPE = "Facility point locations"
SOURCE_FORMATS = ["CSV", "GeoJSON"]

print("Dataset metadata")
print("-" * 60)
print(f"Title            : {DATASET_TITLE}")
print(f"Provider         : {DATASET_PROVIDER}")
print(f"Dataset year     : {DATASET_YEAR}")
print(f"Reporting unit   : {EMISSIONS_UNIT}")
print(f"Geometry         : {GEOMETRY_TYPE}")
print(f"Source formats   : {', '.join(SOURCE_FORMATS)}")

print("\nSource files")
print("-" * 60)
print(f"CSV      : {CO2_CSV_PATH.name}")
print(f"GeoJSON  : {CO2_JSON_PATH.name}")

Dataset metadata
------------------------------------------------------------
Title            : Greenhouse Gas Emissions from Large Facilities
Provider         : Environment and Climate Change Canada (ECCC)
Dataset year     : 2024
Reporting unit   : kt CO2e/year
Geometry         : Facility point locations
Source formats   : CSV, GeoJSON

Source files
------------------------------------------------------------
CSV      : Greenhouse gas emissions from large facilities - 2024.csv
GeoJSON  : AirEmissions_GHG_2024.json


In [4]:
## =============================================================================
# Dataset schema audit
# =============================================================================

print("CSV schema")
print("-" * 60)
print(f"Rows    : {len(co2_csv):,}")
print(f"Columns : {len(co2_csv.columns)}")

display(
    pd.DataFrame(
        {
            "Column": co2_csv.columns,
            "Data Type": co2_csv.dtypes.astype(str).values,
        }
    )
)

print("\nGeoJSON schema")
print("-" * 60)
print(f"Rows    : {len(co2_geo):,}")
print(f"Columns : {len(co2_geo.columns)}")
print(f"CRS     : {co2_geo.crs}")

display(
    pd.DataFrame(
        {
            "Column": co2_geo.columns,
            "Data Type": co2_geo.dtypes.astype(str).values,
        }
    )
)

CSV schema
------------------------------------------------------------
Rows    : 1,879
Columns : 18


,Column,Data Type
0,Facility ID,int64
1,Facility name,object
2,Company name,object
3,City,object
4,Address,object
5,Postal code,object
6,Province,object
7,Latitude,float64
8,Longitude,float64
9,Total emissions,float64



GeoJSON schema
------------------------------------------------------------
Rows    : 1,879
Columns : 13
CRS     : EPSG:4326


,Column,Data Type
0,Name,object
1,Company,object
2,Facility type,object
3,City,object
4,Province,object
5,Latitude,float64
6,Longitude,float64
7,Total GHG emissions,float64
8,Units,object
9,Year,int32


In [5]:
# =============================================================================
# Define canonical column mapping
# =============================================================================

CO2_COLUMN_MAP = {
    "Facility ID": "facility_id",
    "Facility name": "facility_name",
    "Company name": "company_name",
    "City": "city",
    "Address": "address",
    "Postal code": "postal_code",
    "Province": "province",
    "Latitude": "latitude",
    "Longitude": "longitude",
    "Total emissions": "emissions_kt_co2e_per_year",
    "Unit": "source_unit",
    "Year": "year",
    "Report year": "report_year",
    "Industry classification": "industry_classification",
    "Industry classification link": "industry_classification_link",
    "Facility information": "facility_information",
    "Facility details": "facility_details",
    "More information": "more_information",
}


# -----------------------------------------------------------------------------
# Validate source schema
# -----------------------------------------------------------------------------

expected_columns = set(CO2_COLUMN_MAP)
actual_columns = set(co2_csv.columns)

missing_columns = sorted(expected_columns - actual_columns)
extra_columns = sorted(actual_columns - expected_columns)

if missing_columns:
    raise ValueError(
        f"Missing expected CO2 CSV columns: {missing_columns}"
    )

if extra_columns:
    print(f"Additional source columns detected: {extra_columns}")
else:
    print("✓ All source columns accounted for.")


# -----------------------------------------------------------------------------
# Create column audit
# -----------------------------------------------------------------------------

column_audit = pd.DataFrame(
    {
        "original_column": co2_csv.columns,
        "standardized_column": [
            CO2_COLUMN_MAP.get(col, "")
            for col in co2_csv.columns
        ],
        "dtype": co2_csv.dtypes.astype(str).values,
    }
)

display(column_audit)


# -----------------------------------------------------------------------------
# Standardize column names
# -----------------------------------------------------------------------------

co2 = co2_csv.rename(columns=CO2_COLUMN_MAP).copy()

print("\nStandardized CO2 dataframe created")
print(f"  Rows:    {len(co2):,}")
print(f"  Columns: {len(co2.columns):,}")

display(co2.head())

✓ All source columns accounted for.


,original_column,standardized_column,dtype
0,Facility ID,facility_id,int64
1,Facility name,facility_name,object
2,Company name,company_name,object
3,City,city,object
4,Address,address,object
5,Postal code,postal_code,object
6,Province,province,object
7,Latitude,latitude,float64
8,Longitude,longitude,float64
9,Total emissions,emissions_kt_co2e_per_year,float64



Standardized CO2 dataframe created
  Rows:    1,879
  Columns: 18


,facility_id,facility_name,company_name,city,address,postal_code,province,latitude,longitude,emissions_kt_co2e_per_year,source_unit,year,report_year,industry_classification,industry_classification_link,facility_information,facility_details,more_information
0,10001,Division Alma,Produits forestiers Résolu,Alma,1100 Melançon Street,G8B 5W2,Quebec,48.56500,-71.65556,53.34,kilotonnes of carbon dioxide equivalents (kt C...,2024,2026,Mechanical Pulp Mills,https://www23.statcan.gc.ca/imdb/p3VD.pl?Funct...,https://climate-change.canada.ca/facility-emis...,NaN,https://indicators-map.dev.ec.gc.ca/App/Detail...
1,10003,"Foothills Pipeline, Alberta",Foothills Pipe Lines Ltd.,Airdrie,NaN,T4A 2G7,Alberta,0.00000,0.00000,254.60,kilotonnes of carbon dioxide equivalents (kt C...,2024,2026,Pipeline Transportation of Natural Gas,https://www23.statcan.gc.ca/imdb/p3VD.pl?Funct...,https://climate-change.canada.ca/facility-emis...,NaN,https://indicators-map.dev.ec.gc.ca/App/Detail...
2,10004,Kingston CoGen,Kingston CoGen Limited Partnership,Bath,5146 Taylor-Kidd Boulevard,K0H 1G0,Ontario,44.20950,-76.72460,1.68,kilotonnes of carbon dioxide equivalents (kt C...,2024,2026,Fossil-Fuel Electric Power Generation,https://www23.statcan.gc.ca/imdb/p3VD.pl?Funct...,https://climate-change.canada.ca/facility-emis...,NaN,https://indicators-map.dev.ec.gc.ca/App/Detail...
3,10006,Redwater Fertilizer Operations,Nutrien (Canada) Holdings ULC,Sturgeon County,56225 SH643,T0A 2W0,Alberta,53.84200,-113.09300,1095.80,kilotonnes of carbon dioxide equivalents (kt C...,2024,2026,Chemical Fertilizer (except Potash) Manufacturing,https://www23.statcan.gc.ca/imdb/p3VD.pl?Funct...,https://climate-change.canada.ca/facility-emis...,NaN,https://indicators-map.dev.ec.gc.ca/App/Detail...
4,10007,Alberta Envirofuels,Keyera Corp,Edmonton,9511 17 Street Northwest,T6P 1Y3,Alberta,53.53199,-113.36492,315.20,kilotonnes of carbon dioxide equivalents (kt C...,2024,2026,Petrochemical Manufacturing,https://www23.statcan.gc.ca/imdb/p3VD.pl?Funct...,https://climate-change.canada.ca/facility-emis...,NaN,https://indicators-map.dev.ec.gc.ca/App/Detail...


In [6]:
# =============================================================================
# Validate standardized CO2 source table
# =============================================================================

# -----------------------------------------------------------------------------
# Convert key columns to numeric
# -----------------------------------------------------------------------------

numeric_columns = [
    "latitude",
    "longitude",
    "emissions_kt_co2e_per_year",
    "year",
    "report_year",
]

for col in numeric_columns:
    co2[col] = pd.to_numeric(co2[col], errors="coerce")


# -----------------------------------------------------------------------------
# Basic unit and year checks
# -----------------------------------------------------------------------------

print("Reported units:")
display(co2["source_unit"].value_counts(dropna=False).rename("count"))

print("\nReported data years:")
display(co2["year"].value_counts(dropna=False).sort_index().rename("count"))

print("\nReported publication/report years:")
display(co2["report_year"].value_counts(dropna=False).sort_index().rename("count"))


# -----------------------------------------------------------------------------
# Data quality checks
# -----------------------------------------------------------------------------

checks = {
    "missing_facility_id": co2["facility_id"].isna().sum(),
    "missing_facility_name": co2["facility_name"].isna().sum(),
    "missing_latitude": co2["latitude"].isna().sum(),
    "missing_longitude": co2["longitude"].isna().sum(),
    "missing_emissions": co2["emissions_kt_co2e_per_year"].isna().sum(),
    "negative_emissions": (co2["emissions_kt_co2e_per_year"] < 0).sum(),
    "zero_emissions": (co2["emissions_kt_co2e_per_year"] == 0).sum(),
    "duplicate_facility_id": co2["facility_id"].duplicated().sum(),
}

display(pd.Series(checks, name="count").to_frame())


# -----------------------------------------------------------------------------
# Coordinate sanity checks for Canada-scale data
# -----------------------------------------------------------------------------

coord_check = co2[
    ~co2["latitude"].between(40, 85)
    | ~co2["longitude"].between(-145, -45)
].copy()

print(f"Rows with coordinates outside broad Canada bounds: {len(coord_check):,}")

if len(coord_check) > 0:
    display(
        coord_check[
            [
                "facility_id",
                "facility_name",
                "company_name",
                "province",
                "city",
                "industry_classification",
                "latitude",
                "longitude",
                "emissions_kt_co2e_per_year",
            ]
        ]
    )

Reported units:


source_unit
kilotonnes of carbon dioxide equivalents (kt CO2 eq)    1879
Name: count, dtype: int64


Reported data years:


year
2024    1879
Name: count, dtype: int64


Reported publication/report years:


report_year
2026    1879
Name: count, dtype: int64

,count
missing_facility_id,0
missing_facility_name,0
missing_latitude,0
missing_longitude,0
missing_emissions,0
negative_emissions,0
zero_emissions,7
duplicate_facility_id,0


Rows with coordinates outside broad Canada bounds: 16


,facility_id,facility_name,company_name,province,city,industry_classification,latitude,longitude,emissions_kt_co2e_per_year
1,10003,"Foothills Pipeline, Alberta",Foothills Pipe Lines Ltd.,Alberta,Airdrie,Pipeline Transportation of Natural Gas,0.0,0.0,254.60
5,10008,Alliance Pipeline Ltd. - AB Pipeline System,Alliance Pipeline Ltd. Partnership,Alberta,Calgary,Pipeline Transportation of Natural Gas,0.0,0.0,535.91
12,10015,ATCO Pipelines - Transmission System,ATCO Gas and Pipelines Ltd.,Alberta,Edmonton,Pipeline Transportation of Natural Gas,0.0,0.0,104.52
29,10038,"TransCanada Pipeline, Saskatchewan",TransCanada PipeLines Limited,Saskatchewan,Burstall,Pipeline Transportation of Natural Gas,0.0,0.0,1200.67
37,10046,"TransCanada Pipeline, Alberta System",Nova Gas Transmission Ltd.,Alberta,Fairview,Pipeline Transportation of Natural Gas,0.0,0.0,4929.01
68,10086,ATCO Gas - Distribution System,ATCO Gas and Pipelines Ltd.,Alberta,Edmonton,Natural Gas Distribution,0.0,0.0,138.10
129,10163,"TransCanada Pipeline, Ontario",TransCanada PipeLines Limited,Ontario,Kenora,Pipeline Transportation of Natural Gas,0.0,0.0,1567.89
185,10244,"TransCanada Pipeline, Manitoba",TransCanada PipeLines Limited,Manitoba,Winnipeg,Pipeline Transportation of Natural Gas,0.0,0.0,312.70
188,10248,"Foothills Pipeline, Saskatchewan",Foothills Pipe Lines (Sask.) Ltd.,Saskatchewan,Richmond,Pipeline Transportation of Natural Gas,0.0,0.0,113.89
197,10257,Alliance Pipeline Ltd. - SK Pipeline System,Alliance Pipeline Ltd. Partnership,Saskatchewan,Calgary,Pipeline Transportation of Natural Gas,0.0,0.0,418.28


In [7]:
# =============================================================================
# Inspect facilities with missing or invalid coordinates
# =============================================================================

invalid_coordinates = co2[
    (co2["latitude"] == 0)
    | (co2["longitude"] == 0)
].copy()

print(f"Facilities with (0,0) coordinates: {len(invalid_coordinates):,}")

display(
    invalid_coordinates[
        [
            "facility_id",
            "facility_name",
            "company_name",
            "province",
            "city",
            "industry_classification",
            "emissions_kt_co2e_per_year",
        ]
    ].sort_values("emissions_kt_co2e_per_year", ascending=False)
)

print("\nIndustry classifications of facilities with invalid coordinates:")
display(
    invalid_coordinates["industry_classification"]
    .value_counts()
    .rename("count")
)

print(
    "\nTotal emissions with invalid coordinates "
    f"(kt CO2e/year): {invalid_coordinates['emissions_kt_co2e_per_year'].sum():,.2f}"
)

Facilities with (0,0) coordinates: 16


,facility_id,facility_name,company_name,province,city,industry_classification,emissions_kt_co2e_per_year
37,10046,"TransCanada Pipeline, Alberta System",Nova Gas Transmission Ltd.,Alberta,Fairview,Pipeline Transportation of Natural Gas,4929.01
129,10163,"TransCanada Pipeline, Ontario",TransCanada PipeLines Limited,Ontario,Kenora,Pipeline Transportation of Natural Gas,1567.89
29,10038,"TransCanada Pipeline, Saskatchewan",TransCanada PipeLines Limited,Saskatchewan,Burstall,Pipeline Transportation of Natural Gas,1200.67
5,10008,Alliance Pipeline Ltd. - AB Pipeline System,Alliance Pipeline Ltd. Partnership,Alberta,Calgary,Pipeline Transportation of Natural Gas,535.91
197,10257,Alliance Pipeline Ltd. - SK Pipeline System,Alliance Pipeline Ltd. Partnership,Saskatchewan,Calgary,Pipeline Transportation of Natural Gas,418.28
185,10244,"TransCanada Pipeline, Manitoba",TransCanada PipeLines Limited,Manitoba,Winnipeg,Pipeline Transportation of Natural Gas,312.70
1,10003,"Foothills Pipeline, Alberta",Foothills Pipe Lines Ltd.,Alberta,Airdrie,Pipeline Transportation of Natural Gas,254.60
224,10290,TransGas Limited,TransGas Limited,Saskatchewan,Regina,Pipeline Transportation of Natural Gas,184.76
68,10086,ATCO Gas - Distribution System,ATCO Gas and Pipelines Ltd.,Alberta,Edmonton,Natural Gas Distribution,138.10
188,10248,"Foothills Pipeline, Saskatchewan",Foothills Pipe Lines (Sask.) Ltd.,Saskatchewan,Richmond,Pipeline Transportation of Natural Gas,113.89



Industry classifications of facilities with invalid coordinates:


industry_classification
Pipeline Transportation of Natural Gas    12
Natural Gas Distribution                   4
Name: count, dtype: int64


Total emissions with invalid coordinates (kt CO2e/year): 9,948.32


In [9]:
# =============================================================================
# Create spatial emissions dataset
# =============================================================================

# =============================================================================
# Flag spatially assignable facilities
# =============================================================================

co2["is_spatially_assignable"] = (
    co2["latitude"].between(40, 85)
    & co2["longitude"].between(-145, -45)
    & ~((co2["latitude"] == 0) | (co2["longitude"] == 0))
)

print("Spatial assignment summary")
display(
    co2["is_spatially_assignable"]
    .value_counts()
    .rename_axis("is_spatially_assignable")
    .to_frame("count")
)

print(f"\nSpatially assignable facilities: {co2['is_spatially_assignable'].sum():,}")
print(f"Non-spatial facilities: {(~co2['is_spatially_assignable']).sum():,}")

# -----------------------------------------------------------------------------
# Filter spatially assignable facilities
# -----------------------------------------------------------------------------

co2_spatial = co2.loc[co2["is_spatially_assignable"]].copy()


# -----------------------------------------------------------------------------
# Convert to GeoDataFrame
# -----------------------------------------------------------------------------

co2_spatial = gpd.GeoDataFrame(
    co2_spatial,
    geometry=gpd.points_from_xy(
        co2_spatial["longitude"],
        co2_spatial["latitude"],
    ),
    crs="EPSG:4326",
)


# -----------------------------------------------------------------------------
# Summary
# -----------------------------------------------------------------------------

print("Spatial emissions dataset")
print("-" * 60)
print(f"Facilities              : {len(co2_spatial):,}")
print(f"Total emissions (kt)    : {co2_spatial['emissions_kt_co2e_per_year'].sum():,.2f}")
print(f"Coordinate reference    : {co2_spatial.crs}")

display(co2_spatial.head())

Spatial assignment summary


,count
is_spatially_assignable,
True,1863
False,16



Spatially assignable facilities: 1,863
Non-spatial facilities: 16
Spatial emissions dataset
------------------------------------------------------------
Facilities              : 1,863
Total emissions (kt)    : 281,713.94
Coordinate reference    : EPSG:4326


,facility_id,facility_name,company_name,city,address,postal_code,province,latitude,longitude,emissions_kt_co2e_per_year,source_unit,year,report_year,industry_classification,industry_classification_link,facility_information,facility_details,more_information,is_spatially_assignable,geometry
0,10001,Division Alma,Produits forestiers Résolu,Alma,1100 Melançon Street,G8B 5W2,Quebec,48.56500,-71.65556,53.34,kilotonnes of carbon dioxide equivalents (kt C...,2024,2026,Mechanical Pulp Mills,https://www23.statcan.gc.ca/imdb/p3VD.pl?Funct...,https://climate-change.canada.ca/facility-emis...,NaN,https://indicators-map.dev.ec.gc.ca/App/Detail...,True,POINT (-71.65556 48.565)
2,10004,Kingston CoGen,Kingston CoGen Limited Partnership,Bath,5146 Taylor-Kidd Boulevard,K0H 1G0,Ontario,44.20950,-76.72460,1.68,kilotonnes of carbon dioxide equivalents (kt C...,2024,2026,Fossil-Fuel Electric Power Generation,https://www23.statcan.gc.ca/imdb/p3VD.pl?Funct...,https://climate-change.canada.ca/facility-emis...,NaN,https://indicators-map.dev.ec.gc.ca/App/Detail...,True,POINT (-76.7246 44.2095)
3,10006,Redwater Fertilizer Operations,Nutrien (Canada) Holdings ULC,Sturgeon County,56225 SH643,T0A 2W0,Alberta,53.84200,-113.09300,1095.80,kilotonnes of carbon dioxide equivalents (kt C...,2024,2026,Chemical Fertilizer (except Potash) Manufacturing,https://www23.statcan.gc.ca/imdb/p3VD.pl?Funct...,https://climate-change.canada.ca/facility-emis...,NaN,https://indicators-map.dev.ec.gc.ca/App/Detail...,True,POINT (-113.093 53.842)
4,10007,Alberta Envirofuels,Keyera Corp,Edmonton,9511 17 Street Northwest,T6P 1Y3,Alberta,53.53199,-113.36492,315.20,kilotonnes of carbon dioxide equivalents (kt C...,2024,2026,Petrochemical Manufacturing,https://www23.statcan.gc.ca/imdb/p3VD.pl?Funct...,https://climate-change.canada.ca/facility-emis...,NaN,https://indicators-map.dev.ec.gc.ca/App/Detail...,True,POINT (-113.36492 53.53199)
6,10009,Alberta-Pacific Forest Industries Inc.,Alberta-Pacific Forest Industries Inc.,County of Athabasca,NaN,T0A 0M0,Alberta,54.92312,-112.86187,144.81,kilotonnes of carbon dioxide equivalents (kt C...,2024,2026,Chemical Pulp Mills,https://www23.statcan.gc.ca/imdb/p3VD.pl?Funct...,https://climate-change.canada.ca/facility-emis...,NaN,https://indicators-map.dev.ec.gc.ca/App/Detail...,True,POINT (-112.86187 54.92312)


In [10]:
# =============================================================================
# Export validated CO2 facility datasets
# =============================================================================

co2.to_csv(CO2_CLEAN_CSV, index=False)

co2_spatial.to_file(
    CO2_CLEAN_GPKG,
    layer="co2_large_facilities_2024",
    driver="GPKG",
)

print("Export complete:")
print(f"  ✓ CSV:  {CO2_CLEAN_CSV}")
print(f"  ✓ GPKG: {CO2_CLEAN_GPKG}")

column_audit.to_csv(CO2_COLUMN_AUDIT_CSV, index=False)

metadata = pd.DataFrame(
    {
        "field": [
            "dataset_title",
            "provider",
            "dataset_year",
            "emissions_unit",
            "spatial_facilities",
            "non_spatial_facilities",
            "spatial_emissions_kt_co2e",
        ],
        "value": [
            DATASET_TITLE,
            DATASET_PROVIDER,
            DATASET_YEAR,
            EMISSIONS_UNIT,
            len(co2_spatial),
            (~co2["is_spatially_assignable"]).sum(),
            co2_spatial["emissions_kt_co2e_per_year"].sum(),
        ],
    }
)

metadata.to_csv(CO2_METADATA_CSV, index=False)

Export complete:
  ✓ CSV:  C:\Users\aviga\Research\repos\temoa_geospace\data_files\processed\emissions\co2_large_facilities_2024\co2_large_facilities_2024_clean.csv
  ✓ GPKG: C:\Users\aviga\Research\repos\temoa_geospace\data_files\processed\emissions\co2_large_facilities_2024\co2_large_facilities_2024_clean.gpkg
